# 04 · SQL Analysis & KPI Development

**Project:** Data Analyst – Mental Health (Canada) · **Pipeline steps 4–5**

This notebook loads the **cleaned** tables from `data/processed/`, registers them in an
in-memory **SQLite** database, and runs a battery of SQL questions that feed the KPI list
and the dashboard.

> **The cleaned data is not in `data/processed/` yet** (the team builds it in
> `02_data_cleaning.ipynb`). Until then this notebook runs on a small **sample dataset**
> generated in section 2 so every query is executable and you can see the output shape.
> When the real files land, the loader picks them up automatically — nothing to change.

---

## Data contract — what `02_data_cleaning.ipynb` should output

**`data/processed/mh_long.csv`** — all five StatCan tables melted into one tidy table:

| column | type | notes |
|---|---|---|
| `source` | text | `perceived_mh_annual` · `suicidal_thoughts` · `stress_coping` · `perceived_health_quarterly` · `cchs_mh_disorders` |
| `geo` | text | canonical name: `Canada`, `Ontario`, `Quebec`, … |
| `geo_level` | text | `national` · `province` · `territory` · `region` |
| `period` | text | original label: `2019/2020`, `2022`, `2021-04` |
| `year` | int | numeric year for ordering (end year of a range) |
| `age_group` | text | `Total, 18 years and over`, `12 to 17 years`, … |
| `sex` | text | `Both` · `Male` · `Female` |
| `indicator` | text | StatCan indicator label (see `docs/data_dictionary.md` §B) |
| `measure` | text | `percent` · `number` |
| `value` | real | the estimate (NULL if suppressed) |
| `ci_low`, `ci_high` | real | 95% CI bounds (nullable) |
| `quality_flag` | text | `''` · `E` · `F` · `x` · `..` |

**`data/processed/cihi_children.csv`** — the two CIHI hidden sheets, tidied:

| column | type | notes |
|---|---|---|
| `service` | text | `ED visit` · `hospitalisation` |
| `fiscal_year` | text | `2018-2019` … `2023-2024` |
| `year` | int | end year |
| `diagnosis_category` | text | e.g. `Mood disorders` |
| `sex` | text | `Female` · `Male` · `Total` |
| `age_group` | text | `5-9` · `10-14` · `15-17` |
| `rate_per_100k` | real | |
| `ci_low`, `ci_high` | real | |

If your column names differ, either rename in cleaning **or** edit the `COLS` map in section 2.


## 1 · Setup

In [1]:
import sqlite3
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 60)

print("sqlite3 runtime:", sqlite3.sqlite_version, "(window functions need >= 3.25)")


sqlite3 runtime: 3.53.4 (window functions need >= 3.25)


In [2]:
def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "data" / "processed").is_dir():
            return p
    raise FileNotFoundError("Could not find data/processed above " + str(start))

ROOT = find_root(Path.cwd())
PROCESSED = ROOT / "data" / "processed"
print("processed folder:", PROCESSED)
print("files present   :", sorted(f.name for f in PROCESSED.glob("*.csv")) or "(none yet)")


processed folder: /Users/samir/Work Related/JDA-Scrum/Data-Analyst-Mental-Health-Project-/data/processed
files present   : ['01_column_profiles.csv', '01_data_inventory.csv']


## 2 · Load cleaned data → SQLite

Looks for `mh_long.csv` / `cihi_children.csv` in `data/processed/`.
If they are not there yet, builds a small sample so the notebook still runs.

In [3]:
# expected files -> table names
EXPECTED = {
    "mh_long":        "mh_long.csv",
    "cihi_children":  "cihi_children.csv",
}

# if the cleaning team uses different column names, map them here (clean_name: their_name)
COLS = {
    # "value": "VALUE", "geo": "GEO", ...
}

def _standardise(df: pd.DataFrame) -> pd.DataFrame:
    if COLS:
        df = df.rename(columns={v: k for k, v in COLS.items()})
    return df

def build_sample():
    """Small synthetic stand-in matching the data contract. DELETE-safe: ignored once real files exist."""
    rng = np.random.default_rng(42)
    geos = {"Canada": "national",
            "Ontario": "province", "Quebec": "province", "British Columbia": "province",
            "Alberta": "province", "Manitoba": "province", "Nova Scotia": "province",
            "New Brunswick": "province", "Saskatchewan": "province",
            "Newfoundland and Labrador": "province", "Prince Edward Island": "province",
            "Yukon": "territory", "Northwest Territories": "territory", "Nunavut": "territory"}
    periods = [("2019/2020", 2020), ("2021/2022", 2022), ("2023/2024", 2024)]
    indicators = {
        "perceived_mh_annual": [
            "Perceived mental health, fair or poor",
            "Perceived mental health, very good or excellent",
            "Perceived life stress, most days quite a bit or extremely stressful",
            "Mood disorder", "Anxiety disorder",
            "Sense of belonging to local community, somewhat strong or very strong"],
        "suicidal_thoughts": [
            "Suicidal thoughts (15 years and over)",
            "Consultation with a health professional about emotional or mental health",
            "Positive mental health, flourishing"],
    }
    base = {"Perceived mental health, fair or poor": 11,
            "Perceived mental health, very good or excellent": 68,
            "Perceived life stress, most days quite a bit or extremely stressful": 21,
            "Mood disorder": 9, "Anxiety disorder": 11,
            "Sense of belonging to local community, somewhat strong or very strong": 66,
            "Suicidal thoughts (15 years and over)": 3.2,
            "Consultation with a health professional about emotional or mental health": 12,
            "Positive mental health, flourishing": 72}
    rows = []
    for source, inds in indicators.items():
        for ind in inds:
            for gi, (geo, lvl) in enumerate(geos.items()):
                for pi, (period, yr) in enumerate(periods):
                    for sex in ["Both", "Male", "Female"]:
                        drift = pi * (0.4 if "fair or poor" in ind or "stress" in ind or "Suicidal" in ind else -0.3)
                        sexadj = {"Both": 0, "Female": 1.1, "Male": -1.0}[sex] * (0.4 if "Suicidal" in ind or "Anxiety" in ind or "Mood" in ind else 0.2)
                        val = base[ind] + drift + sexadj + rng.normal(0, 0.8) + (gi % 4) * 0.3
                        val = max(0.1, round(val, 1))
                        rows.append(dict(source=source, geo=geo, geo_level=lvl, period=period, year=yr,
                                         age_group="Total, 18 years and over", sex=sex, indicator=ind,
                                         measure="percent", value=val,
                                         ci_low=round(val - 1.2, 1), ci_high=round(val + 1.2, 1),
                                         quality_flag=""))
    mh_long = pd.DataFrame(rows)

    dx = ["Mood disorders", "Anxiety disorders", "Substance-related disorders",
          "Neurocognitive disorders", "Selected disorders diagnosed in childhood"]
    fys = [("2018-2019", 2019), ("2020-2021", 2021), ("2022-2023", 2023), ("2023-2024", 2024)]
    crows = []
    for service, b in [("ED visit", 1500), ("hospitalisation", 380)]:
        for d in dx:
            for (fy, yr) in fys:
                for sex in ["Female", "Male", "Total"]:
                    for ag in ["5-9", "10-14", "15-17"]:
                        mult = {"5-9": 0.2, "10-14": 0.8, "15-17": 1.6}[ag]
                        sx = {"Female": 1.15, "Male": 0.9, "Total": 1.0}[sex]
                        r = b * mult * sx * (1 + 0.03 * (yr - 2019)) * rng.uniform(0.9, 1.1)
                        crows.append(dict(service=service, fiscal_year=fy, year=yr, diagnosis_category=d,
                                          sex=sex, age_group=ag, rate_per_100k=round(r, 1),
                                          ci_low=round(r * 0.93, 1), ci_high=round(r * 1.07, 1)))
    return {"mh_long": mh_long, "cihi_children": pd.DataFrame(crows)}


frames, USING_SAMPLE = {}, False
missing = [f for f in EXPECTED.values() if not (PROCESSED / f).exists()]
if missing:
    USING_SAMPLE = True
    print("⚠️  real files not found:", missing, "\n    -> using SAMPLE data so the notebook runs.\n")
    frames = build_sample()
else:
    for table, fname in EXPECTED.items():
        frames[table] = _standardise(pd.read_csv(PROCESSED / fname))
    print("loaded real files:", list(EXPECTED.values()))

con = sqlite3.connect(":memory:")
for table, df in frames.items():
    df.to_sql(table, con, if_exists="replace", index=False)
    print(f"  table {table:16} {df.shape[0]:>5} rows  |  cols: {list(df.columns)}")


⚠️  real files not found: ['mh_long.csv', 'cihi_children.csv'] 
    -> using SAMPLE data so the notebook runs.

  table mh_long           1134 rows  |  cols: ['source', 'geo', 'geo_level', 'period', 'year', 'age_group', 'sex', 'indicator', 'measure', 'value', 'ci_low', 'ci_high', 'quality_flag']
  table cihi_children      360 rows  |  cols: ['service', 'fiscal_year', 'year', 'diagnosis_category', 'sex', 'age_group', 'rate_per_100k', 'ci_low', 'ci_high']


In [4]:
def q(sql: str) -> pd.DataFrame:
    """Run a SQL query against the in-memory DB and return a DataFrame."""
    return pd.read_sql_query(textwrap.dedent(sql), con)

# reference: indicator labels available
q("SELECT source, indicator, COUNT(*) AS rows FROM mh_long GROUP BY 1, 2 ORDER BY 1, 2")


,source,indicator,rows
0,perceived_mh_annual,Anxiety disorder,126
1,perceived_mh_annual,Mood disorder,126
2,perceived_mh_annual,"Perceived life stress, most days quite a bit o...",126
3,perceived_mh_annual,"Perceived mental health, fair or poor",126
4,perceived_mh_annual,"Perceived mental health, very good or excellent",126
5,perceived_mh_annual,"Sense of belonging to local community, somewha...",126
6,suicidal_thoughts,Consultation with a health professional about ...,126
7,suicidal_thoughts,"Positive mental health, flourishing",126
8,suicidal_thoughts,Suicidal thoughts (15 years and over),126


## 3 · Coverage & data-quality checks
_Run these first whenever new data lands._

**Q1 · Row counts and column list per table**

In [5]:
q("SELECT 'mh_long' AS tbl, COUNT(*) n FROM mh_long UNION ALL SELECT 'cihi_children', COUNT(*) FROM cihi_children")

,tbl,n
0,mh_long,1134
1,cihi_children,360


**Q2 · Time coverage per source** — how many periods do we actually have?

In [6]:
q("""
    SELECT source,
           COUNT(DISTINCT period) AS n_periods,
           MIN(year) AS first_year,
           MAX(year) AS last_year,
           GROUP_CONCAT(DISTINCT period) AS periods
    FROM mh_long
    GROUP BY source
""")

,source,n_periods,first_year,last_year,periods
0,perceived_mh_annual,3,2020,2024,"2019/2020,2021/2022,2023/2024"
1,suicidal_thoughts,3,2020,2024,"2019/2020,2021/2022,2023/2024"


**Q3 · Geography coverage** — provinces / territories / regions present

In [7]:
q("""
    SELECT geo_level, COUNT(DISTINCT geo) AS n_geo,
           GROUP_CONCAT(DISTINCT geo) AS geos
    FROM mh_long GROUP BY geo_level ORDER BY n_geo DESC
""")

,geo_level,n_geo,geos
0,province,10,"Ontario,Quebec,British Columbia,Alberta,Manito..."
1,territory,3,"Yukon,Northwest Territories,Nunavut"
2,national,1,Canada


**Q4 · Suppressed / low-quality data** — share of rows by `quality_flag`

In [8]:
q("""
    SELECT source,
           COUNT(*) AS rows,
           SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) AS null_value,
           SUM(CASE WHEN quality_flag IN ('E') THEN 1 ELSE 0 END) AS use_with_caution,
           SUM(CASE WHEN quality_flag IN ('F','x','..') THEN 1 ELSE 0 END) AS suppressed,
           ROUND(100.0 * SUM(CASE WHEN value IS NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_missing
    FROM mh_long GROUP BY source ORDER BY pct_missing DESC
""")

,source,rows,null_value,use_with_caution,suppressed,pct_missing
0,suicidal_thoughts,378,0,0,0,0.0
1,perceived_mh_annual,756,0,0,0,0.0


**Q5 · Duplicate grain check** — should return 0 rows (one value per geo/period/sex/age/indicator)

In [9]:
q("""
    SELECT source, geo, period, sex, age_group, indicator, COUNT(*) AS n
    FROM mh_long
    GROUP BY 1,2,3,4,5,6
    HAVING COUNT(*) > 1
""")

,source,geo,period,sex,age_group,indicator,n


## 4 · Current state — latest-period snapshot

**Q6 · National (Canada) latest value for every indicator** — the headline numbers

In [10]:
q("""
    WITH latest AS (
        SELECT indicator, MAX(year) AS y
        FROM mh_long WHERE geo = 'Canada' AND sex = 'Both'
        GROUP BY indicator
    )
    SELECT m.indicator, m.period, m.value, m.ci_low, m.ci_high, m.quality_flag
    FROM mh_long m
    JOIN latest l ON l.indicator = m.indicator AND l.y = m.year
    WHERE m.geo = 'Canada' AND m.sex = 'Both'
    ORDER BY m.value DESC
""")

,indicator,period,value,ci_low,ci_high,quality_flag
0,"Positive mental health, flourishing",2023/2024,70.6,69.4,71.8,
1,"Perceived mental health, very good or excellent",2023/2024,67.9,66.7,69.1,
2,"Sense of belonging to local community, somewha...",2023/2024,65.2,64.0,66.4,
3,"Perceived life stress, most days quite a bit o...",2023/2024,21.7,20.5,22.9,
4,"Perceived mental health, fair or poor",2023/2024,11.9,10.7,13.1,
5,Consultation with a health professional about ...,2023/2024,9.4,8.2,10.6,
6,Anxiety disorder,2023/2024,9.2,8.0,10.4,
7,Mood disorder,2023/2024,7.1,5.9,8.3,
8,Suicidal thoughts (15 years and over),2023/2024,4.6,3.4,5.8,


**Q7 · Fair/poor perceived mental health — latest, by province, ranked worst → best**  
_Feeds KPI: highest / lowest-burden region._

In [11]:
q("""
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator = 'Perceived mental health, fair or poor')
    SELECT geo, value AS pct_fair_or_poor, ci_low, ci_high,
           RANK() OVER (ORDER BY value DESC) AS worst_rank
    FROM mh_long, latest
    WHERE indicator = 'Perceived mental health, fair or poor'
      AND sex = 'Both' AND geo_level IN ('province','territory')
      AND year = latest.y AND value IS NOT NULL
    ORDER BY value DESC
""")

,geo,pct_fair_or_poor,ci_low,ci_high,worst_rank
0,Nunavut,13.2,12.0,14.4,1
1,Manitoba,12.6,11.4,13.8,2
2,New Brunswick,12.5,11.3,13.7,3
3,Yukon,12.5,11.3,13.7,3
4,Saskatchewan,12.3,11.1,13.5,5
5,Quebec,12.1,10.9,13.3,6
6,Newfoundland and Labrador,12.1,10.9,13.3,6
7,British Columbia,12.0,10.8,13.2,8
8,Northwest Territories,11.9,10.7,13.1,9
9,Ontario,11.4,10.2,12.6,10


**Q8 · Suicidal thoughts — latest, national, by sex**

In [12]:
q("""
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator = 'Suicidal thoughts (15 years and over)')
    SELECT sex, value AS pct, ci_low, ci_high
    FROM mh_long, latest
    WHERE indicator = 'Suicidal thoughts (15 years and over)'
      AND geo = 'Canada' AND year = latest.y
    ORDER BY sex
""")

,sex,pct,ci_low,ci_high
0,Both,4.6,3.4,5.8
1,Female,4.7,3.5,5.9
2,Male,4.1,2.9,5.3


**Q9 · Every province vs the national rate (latest)** — who is above / below Canada

In [13]:
q("""
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator = 'Perceived mental health, fair or poor'),
    nat AS (SELECT value AS canada FROM mh_long, latest
            WHERE indicator = 'Perceived mental health, fair or poor'
              AND geo = 'Canada' AND sex = 'Both' AND year = latest.y)
    SELECT m.geo, m.value, nat.canada,
           ROUND(m.value - nat.canada, 1) AS diff_vs_canada,
           CASE WHEN m.value > nat.canada THEN 'above' ELSE 'at/below' END AS position
    FROM mh_long m, latest, nat
    WHERE m.indicator = 'Perceived mental health, fair or poor'
      AND m.sex = 'Both' AND m.geo_level IN ('province','territory')
      AND m.year = latest.y
    ORDER BY diff_vs_canada DESC
""")

,geo,value,canada,diff_vs_canada,position
0,Nunavut,13.2,11.9,1.3,above
1,Manitoba,12.6,11.9,0.7,above
2,New Brunswick,12.5,11.9,0.6,above
3,Yukon,12.5,11.9,0.6,above
4,Saskatchewan,12.3,11.9,0.4,above
5,Newfoundland and Labrador,12.1,11.9,0.2,above
6,Quebec,12.1,11.9,0.2,above
7,British Columbia,12.0,11.9,0.1,above
8,Northwest Territories,11.9,11.9,0.0,at/below
9,Ontario,11.4,11.9,-0.5,at/below


## 5 · Geography — spread and rankings

**Q10 · Provincial spread per indicator (latest)** — max − min. Where is the inequality largest?  
_Feeds KPI: provincial spread._

In [14]:
q("""
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator)
    SELECT m.indicator,
           ROUND(MIN(m.value), 1) AS min_pct,
           ROUND(MAX(m.value), 1) AS max_pct,
           ROUND(MAX(m.value) - MIN(m.value), 1) AS spread_pp,
           ROUND(AVG(m.value), 1) AS avg_pct
    FROM mh_long m JOIN latest l ON l.indicator = m.indicator AND l.y = m.year
    WHERE m.sex = 'Both' AND m.geo_level = 'province' AND m.value IS NOT NULL
    GROUP BY m.indicator
    ORDER BY spread_pp DESC
""")

,indicator,min_pct,max_pct,spread_pp,avg_pct
0,"Positive mental health, flourishing",70.3,74.5,4.2,72.1
1,"Perceived life stress, most days quite a bit o...",21.1,23.8,2.7,22.3
2,"Perceived mental health, very good or excellent",65.8,68.1,2.3,67.4
3,Mood disorder,7.4,9.7,2.3,8.8
4,Consultation with a health professional about ...,10.9,13.2,2.3,12.1
5,Suicidal thoughts (15 years and over),3.3,5.4,2.1,4.5
6,Anxiety disorder,10.2,12.2,2.0,11.1
7,"Sense of belonging to local community, somewha...",64.8,66.5,1.7,65.9
8,"Perceived mental health, fair or poor",11.1,12.6,1.5,11.9


**Q11 · Best and worst province for each indicator (latest)**

In [15]:
q("""
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator),
    ranked AS (
        SELECT m.indicator, m.geo, m.value,
               ROW_NUMBER() OVER (PARTITION BY m.indicator ORDER BY m.value DESC) AS hi,
               ROW_NUMBER() OVER (PARTITION BY m.indicator ORDER BY m.value ASC)  AS lo
        FROM mh_long m JOIN latest l ON l.indicator = m.indicator AND l.y = m.year
        WHERE m.sex = 'Both' AND m.geo_level = 'province' AND m.value IS NOT NULL
    )
    SELECT indicator,
           MAX(CASE WHEN hi = 1 THEN geo END)   AS highest_geo,
           MAX(CASE WHEN hi = 1 THEN value END)  AS highest_value,
           MAX(CASE WHEN lo = 1 THEN geo END)    AS lowest_geo,
           MAX(CASE WHEN lo = 1 THEN value END)  AS lowest_value
    FROM ranked GROUP BY indicator
""")

,indicator,highest_geo,highest_value,lowest_geo,lowest_value
0,Anxiety disorder,British Columbia,12.2,Saskatchewan,10.2
1,Consultation with a health professional about ...,British Columbia,13.2,Prince Edward Island,10.9
2,Mood disorder,Quebec,9.7,British Columbia,7.4
3,"Perceived life stress, most days quite a bit o...",Nova Scotia,23.8,Prince Edward Island,21.1
4,"Perceived mental health, fair or poor",Manitoba,12.6,Nova Scotia,11.1
5,"Perceived mental health, very good or excellent",Nova Scotia,68.1,Saskatchewan,65.8
6,"Positive mental health, flourishing",New Brunswick,74.5,Saskatchewan,70.3
7,"Sense of belonging to local community, somewha...",Quebec,66.5,Alberta,64.8
8,Suicidal thoughts (15 years and over),Ontario,5.4,Manitoba,3.3


**Q12 · Territories vs provinces** — average of each indicator by geo level (latest)

In [16]:
q("""
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator)
    SELECT m.indicator, m.geo_level, ROUND(AVG(m.value), 1) AS avg_pct, COUNT(*) AS n_geo
    FROM mh_long m JOIN latest l ON l.indicator = m.indicator AND l.y = m.year
    WHERE m.sex = 'Both' AND m.geo_level IN ('province','territory') AND m.value IS NOT NULL
    GROUP BY m.indicator, m.geo_level
    ORDER BY m.indicator, m.geo_level
""")

,indicator,geo_level,avg_pct,n_geo
0,Anxiety disorder,province,11.1,10
1,Anxiety disorder,territory,11.1,3
2,Consultation with a health professional about ...,province,12.1,10
3,Consultation with a health professional about ...,territory,12.2,3
4,Mood disorder,province,8.8,10
5,Mood disorder,territory,9.1,3
6,"Perceived life stress, most days quite a bit o...",province,22.3,10
7,"Perceived life stress, most days quite a bit o...",territory,22.7,3
8,"Perceived mental health, fair or poor",province,11.9,10
9,"Perceived mental health, fair or poor",territory,12.5,3


## 6 · Demographics — sex gaps and age gradients

**Q13 · Female − male gap per indicator, national (latest)**  
_Feeds KPI: sex gap._

In [17]:
q("""
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator)
    SELECT m.indicator,
           MAX(CASE WHEN sex = 'Female' THEN value END) AS female,
           MAX(CASE WHEN sex = 'Male'   THEN value END) AS male,
           ROUND(MAX(CASE WHEN sex = 'Female' THEN value END)
               - MAX(CASE WHEN sex = 'Male'   THEN value END), 1) AS female_minus_male
    FROM mh_long m JOIN latest l ON l.indicator = m.indicator AND l.y = m.year
    WHERE m.geo = 'Canada'
    GROUP BY m.indicator
    ORDER BY ABS(female_minus_male) DESC
""")

,indicator,female,male,female_minus_male
0,"Sense of belonging to local community, somewha...",66.1,64.3,1.8
1,Mood disorder,9.2,7.4,1.8
2,"Positive mental health, flourishing",71.7,70.4,1.3
3,"Perceived life stress, most days quite a bit o...",21.8,20.5,1.3
4,Consultation with a health professional about ...,11.0,9.8,1.2
5,Anxiety disorder,10.9,9.8,1.1
6,"Perceived mental health, fair or poor",12.0,11.3,0.7
7,Suicidal thoughts (15 years and over),4.7,4.1,0.6
8,"Perceived mental health, very good or excellent",66.4,67.0,-0.6


**Q14 · Sex gap in suicidal thoughts, by province (latest)** — where is the gap widest?

In [18]:
q("""
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator = 'Suicidal thoughts (15 years and over)')
    SELECT geo,
           MAX(CASE WHEN sex = 'Female' THEN value END) AS female,
           MAX(CASE WHEN sex = 'Male'   THEN value END) AS male,
           ROUND(MAX(CASE WHEN sex = 'Female' THEN value END)
               - MAX(CASE WHEN sex = 'Male'   THEN value END), 1) AS gap
    FROM mh_long, latest
    WHERE indicator = 'Suicidal thoughts (15 years and over)'
      AND geo_level IN ('province','territory') AND year = latest.y
    GROUP BY geo
    ORDER BY gap DESC
""")

,geo,female,male,gap
0,Quebec,6.5,4.1,2.4
1,Northwest Territories,5.0,2.8,2.2
2,Ontario,5.8,4.1,1.7
3,Nunavut,5.5,4.1,1.4
4,Saskatchewan,4.2,2.9,1.3
5,Manitoba,4.9,3.6,1.3
6,Prince Edward Island,5.3,4.2,1.1
7,Newfoundland and Labrador,5.0,4.0,1.0
8,Yukon,5.4,4.5,0.9
9,Nova Scotia,5.3,4.4,0.9


**Q15 · Age gradient** — value by `age_group` where age detail exists (national, latest).  
_Only returns rows once a source with age breakdowns (e.g. suicidal_thoughts, stress_coping) is loaded._

In [19]:
q("""
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator)
    SELECT m.indicator, m.age_group, ROUND(AVG(m.value), 1) AS pct
    FROM mh_long m JOIN latest l ON l.indicator = m.indicator AND l.y = m.year
    WHERE m.geo = 'Canada' AND m.sex = 'Both'
      AND m.age_group <> 'Total, 18 years and over'
    GROUP BY m.indicator, m.age_group
    ORDER BY m.indicator, m.age_group
""")

,indicator,age_group,pct


## 7 · Change over time
_Coverage is thin (2–3 cycles). Treat these as cycle-to-cycle changes, not smooth trends._

**Q16 · National change, first → latest period, per indicator**  
_Feeds KPI: period-over-period change._

In [20]:
q("""
    WITH b AS (
        SELECT indicator, value AS first_val, period AS first_period,
               ROW_NUMBER() OVER (PARTITION BY indicator ORDER BY year ASC)  AS r_first,
               ROW_NUMBER() OVER (PARTITION BY indicator ORDER BY year DESC) AS r_last
        FROM mh_long WHERE geo = 'Canada' AND sex = 'Both'
    )
    SELECT f.indicator, f.first_period, f.first_val,
           l.first_period AS last_period, l.first_val AS last_val,
           ROUND(l.first_val - f.first_val, 1) AS change_pp,
           ROUND(100.0 * (l.first_val - f.first_val) / NULLIF(f.first_val, 0), 1) AS pct_change
    FROM b f JOIN b l ON f.indicator = l.indicator
    WHERE f.r_first = 1 AND l.r_last = 1
    ORDER BY ABS(change_pp) DESC
""")

,indicator,first_period,first_val,last_period,last_val,change_pp,pct_change
0,Mood disorder,2019/2020,10.3,2023/2024,7.1,-3.2,-31.1
1,"Sense of belonging to local community, somewha...",2019/2020,67.3,2023/2024,65.2,-2.1,-3.1
2,Suicidal thoughts (15 years and over),2019/2020,2.5,2023/2024,4.6,2.1,84.0
3,"Positive mental health, flourishing",2019/2020,72.6,2023/2024,70.6,-2.0,-2.8
4,Consultation with a health professional about ...,2019/2020,11.1,2023/2024,9.4,-1.7,-15.3
5,"Perceived mental health, very good or excellent",2019/2020,69.2,2023/2024,67.9,-1.3,-1.9
6,"Perceived life stress, most days quite a bit o...",2019/2020,22.4,2023/2024,21.7,-0.7,-3.1
7,"Perceived mental health, fair or poor",2019/2020,11.2,2023/2024,11.9,0.7,6.3
8,Anxiety disorder,2019/2020,8.6,2023/2024,9.2,0.6,7.0


**Q17 · Period-over-period change per province** (window `LAG`) for fair/poor perceived MH

In [21]:
q("""
    SELECT geo, period, year, value,
           LAG(value)  OVER (PARTITION BY geo ORDER BY year) AS prev_value,
           ROUND(value - LAG(value) OVER (PARTITION BY geo ORDER BY year), 1) AS change_pp
    FROM mh_long
    WHERE indicator = 'Perceived mental health, fair or poor'
      AND sex = 'Both' AND geo_level = 'province'
    ORDER BY geo, year
""")

,geo,period,year,value,prev_value,change_pp
0,Alberta,2019/2020,2020,10.9,NaN,NaN
1,Alberta,2021/2022,2022,11.9,10.9,1.0
2,Alberta,2023/2024,2024,11.3,11.9,-0.6
3,British Columbia,2019/2020,2020,12.2,NaN,NaN
4,British Columbia,2021/2022,2022,14.0,12.2,1.8
5,British Columbia,2023/2024,2024,12.0,14.0,-2.0
6,Manitoba,2019/2020,2020,11.5,NaN,NaN
7,Manitoba,2021/2022,2022,12.2,11.5,0.7
8,Manitoba,2023/2024,2024,12.6,12.2,0.4
9,New Brunswick,2019/2020,2020,12.4,NaN,NaN


**Q18 · % change from the baseline (earliest) period, per province**

In [22]:
q("""
    SELECT geo, period, year, value,
           FIRST_VALUE(value) OVER (PARTITION BY geo ORDER BY year) AS baseline,
           ROUND(100.0 * (value - FIRST_VALUE(value) OVER (PARTITION BY geo ORDER BY year))
                 / NULLIF(FIRST_VALUE(value) OVER (PARTITION BY geo ORDER BY year), 0), 1) AS pct_change_vs_baseline
    FROM mh_long
    WHERE indicator = 'Perceived mental health, fair or poor'
      AND sex = 'Both' AND geo_level = 'province'
    ORDER BY geo, year
""")

,geo,period,year,value,baseline,pct_change_vs_baseline
0,Alberta,2019/2020,2020,10.9,10.9,0.0
1,Alberta,2021/2022,2022,11.9,10.9,9.2
2,Alberta,2023/2024,2024,11.3,10.9,3.7
3,British Columbia,2019/2020,2020,12.2,12.2,0.0
4,British Columbia,2021/2022,2022,14.0,12.2,14.8
5,British Columbia,2023/2024,2024,12.0,12.2,-1.6
6,Manitoba,2019/2020,2020,11.5,11.5,0.0
7,Manitoba,2021/2022,2022,12.2,11.5,6.1
8,Manitoba,2023/2024,2024,12.6,11.5,9.6
9,New Brunswick,2019/2020,2020,12.4,12.4,0.0


**Q19 · How many provinces got worse?** — fair/poor MH higher in latest period than first.  
_Feeds KPI: # provinces with worsening trend._

In [23]:
q("""
    WITH ranked AS (
        SELECT geo, value,
               ROW_NUMBER() OVER (PARTITION BY geo ORDER BY year ASC)  AS r_first,
               ROW_NUMBER() OVER (PARTITION BY geo ORDER BY year DESC) AS r_last
        FROM mh_long
        WHERE indicator = 'Perceived mental health, fair or poor'
          AND sex = 'Both' AND geo_level = 'province'
    ),
    chg AS (
        SELECT f.geo, f.value AS first_val, l.value AS last_val, l.value - f.value AS delta
        FROM ranked f JOIN ranked l ON f.geo = l.geo
        WHERE f.r_first = 1 AND l.r_last = 1
    )
    SELECT
        SUM(CASE WHEN delta > 0 THEN 1 ELSE 0 END) AS provinces_worsening,
        SUM(CASE WHEN delta < 0 THEN 1 ELSE 0 END) AS provinces_improving,
        SUM(CASE WHEN delta = 0 THEN 1 ELSE 0 END) AS provinces_no_change,
        COUNT(*) AS provinces_total
    FROM chg
""")

,provinces_worsening,provinces_improving,provinces_no_change,provinces_total
0,6,4,0,10


**Q20 · Biggest movers** — largest increase and decrease by province, fair/poor MH

In [24]:
q("""
    WITH ranked AS (
        SELECT geo, value, period,
               ROW_NUMBER() OVER (PARTITION BY geo ORDER BY year ASC)  AS r_first,
               ROW_NUMBER() OVER (PARTITION BY geo ORDER BY year DESC) AS r_last
        FROM mh_long
        WHERE indicator = 'Perceived mental health, fair or poor'
          AND sex = 'Both' AND geo_level = 'province'
    )
    SELECT f.geo, f.period AS first_period, f.value AS first_val,
           l.period AS last_period, l.value AS last_val,
           ROUND(l.value - f.value, 1) AS change_pp
    FROM ranked f JOIN ranked l ON f.geo = l.geo
    WHERE f.r_first = 1 AND l.r_last = 1
    ORDER BY change_pp DESC
""")

,geo,first_period,first_val,last_period,last_val,change_pp
0,Saskatchewan,2019/2020,10.3,2023/2024,12.3,2.0
1,Newfoundland and Labrador,2019/2020,10.8,2023/2024,12.1,1.3
2,Manitoba,2019/2020,11.5,2023/2024,12.6,1.1
3,Ontario,2019/2020,10.6,2023/2024,11.4,0.8
4,Alberta,2019/2020,10.9,2023/2024,11.3,0.4
5,New Brunswick,2019/2020,12.4,2023/2024,12.5,0.1
6,Nova Scotia,2019/2020,11.2,2023/2024,11.1,-0.1
7,British Columbia,2019/2020,12.2,2023/2024,12.0,-0.2
8,Quebec,2019/2020,12.3,2023/2024,12.1,-0.2
9,Prince Edward Island,2019/2020,12.1,2023/2024,11.3,-0.8


## 8 · Cross-indicator relationships

**Q21 · Help-seeking gap** — suicidal thoughts vs consultation with a professional, by province (latest).  
_Feeds KPI: help-seeking ratio / unmet need proxy._

In [25]:
q("""
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator)
    SELECT s.geo,
           s.value AS suicidal_thoughts_pct,
           cca.value AS consulted_professional_pct,
           ROUND(cca.value / NULLIF(s.value, 0), 2) AS consult_per_ideation
    FROM mh_long s
    JOIN latest ls ON ls.indicator = s.indicator AND ls.y = s.year
    JOIN mh_long cca ON cca.geo = s.geo AND cca.sex = s.sex
    JOIN latest lc ON lc.indicator = cca.indicator AND lc.y = cca.year
    WHERE s.indicator = 'Suicidal thoughts (15 years and over)'
      AND cca.indicator = 'Consultation with a health professional about emotional or mental health'
      AND s.sex = 'Both' AND s.geo_level IN ('province','territory')
    ORDER BY consult_per_ideation ASC
""")

,geo,suicidal_thoughts_pct,consulted_professional_pct,consult_per_ideation
0,Ontario,5.4,11.0,2.04
1,Prince Edward Island,5.3,10.9,2.06
2,Alberta,4.8,11.3,2.35
3,Quebec,4.6,11.8,2.57
4,New Brunswick,4.9,12.7,2.59
5,British Columbia,4.8,13.2,2.75
6,Newfoundland and Labrador,4.2,11.7,2.79
7,Nunavut,4.3,12.0,2.79
8,Northwest Territories,4.2,12.0,2.86
9,Nova Scotia,4.3,12.4,2.88


**Q22 · Life stress vs fair/poor mental health** — rank provinces on both, compare (latest)

In [26]:
q("""
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator),
    stress AS (
        SELECT geo, value,
               RANK() OVER (ORDER BY value DESC) AS stress_rank
        FROM mh_long m JOIN latest l ON l.indicator = m.indicator AND l.y = m.year
        WHERE m.indicator = 'Perceived life stress, most days quite a bit or extremely stressful'
          AND m.sex = 'Both' AND m.geo_level = 'province'
    ),
    poor AS (
        SELECT geo, value,
               RANK() OVER (ORDER BY value DESC) AS poor_rank
        FROM mh_long m JOIN latest l ON l.indicator = m.indicator AND l.y = m.year
        WHERE m.indicator = 'Perceived mental health, fair or poor'
          AND m.sex = 'Both' AND m.geo_level = 'province'
    )
    SELECT s.geo, s.value AS life_stress_pct, s.stress_rank,
           p.value AS fair_poor_mh_pct, p.poor_rank,
           s.stress_rank - p.poor_rank AS rank_diff
    FROM stress s JOIN poor p ON s.geo = p.geo
    ORDER BY s.stress_rank
""")

,geo,life_stress_pct,stress_rank,fair_poor_mh_pct,poor_rank,rank_diff
0,Nova Scotia,23.8,1,11.1,10,-9
1,Saskatchewan,23.0,2,12.3,3,-1
2,Newfoundland and Labrador,22.6,3,12.1,4,-1
3,New Brunswick,22.5,4,12.5,2,2
4,Ontario,22.4,5,11.4,7,-2
5,British Columbia,22.3,6,12.0,6,0
6,Quebec,21.8,7,12.1,4,3
7,Manitoba,21.8,7,12.6,1,6
8,Alberta,21.6,9,11.3,8,1
9,Prince Edward Island,21.1,10,11.3,8,2


**Q23 · Sense of belonging vs fair/poor mental health** — provinces with high belonging AND low poor-MH (latest)

In [27]:
q("""
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator)
    SELECT b.geo,
           b.value AS belonging_pct,
           p.value AS fair_poor_mh_pct
    FROM mh_long b
    JOIN latest lb ON lb.indicator = b.indicator AND lb.y = b.year
    JOIN mh_long p ON p.geo = b.geo AND p.sex = b.sex
    JOIN latest lp ON lp.indicator = p.indicator AND lp.y = p.year
    WHERE b.indicator = 'Sense of belonging to local community, somewhat strong or very strong'
      AND p.indicator = 'Perceived mental health, fair or poor'
      AND b.sex = 'Both' AND b.geo_level = 'province'
    ORDER BY b.value DESC
""")

,geo,belonging_pct,fair_poor_mh_pct
0,Quebec,66.5,12.1
1,Nova Scotia,66.3,11.1
2,Manitoba,66.2,12.6
3,Newfoundland and Labrador,66.2,12.1
4,Prince Edward Island,66.2,11.3
5,New Brunswick,66.1,12.5
6,British Columbia,66.0,12.0
7,Ontario,65.6,11.4
8,Saskatchewan,65.2,12.3
9,Alberta,64.8,11.3


## 9 · Children & youth (CIHI)

**Q24 · ED-visit rate for mental disorders by fiscal year** (Total sex, all diagnoses) — trend

In [28]:
q("""
    SELECT fiscal_year, year,
           ROUND(AVG(rate_per_100k), 1) AS avg_rate_per_100k
    FROM cihi_children
    WHERE service = 'ED visit' AND sex = 'Total'
    GROUP BY fiscal_year, year
    ORDER BY year
""")

,fiscal_year,year,avg_rate_per_100k
0,2018-2019,2019,1264.2
1,2020-2021,2021,1419.0
2,2022-2023,2023,1432.2
3,2023-2024,2024,1417.0


**Q25 · Hospitalisation rate by diagnosis category, latest fiscal year, by sex**

In [29]:
q("""
    WITH latest AS (SELECT MAX(year) y FROM cihi_children)
    SELECT diagnosis_category,
           MAX(CASE WHEN sex = 'Female' THEN rate_per_100k END) AS female,
           MAX(CASE WHEN sex = 'Male'   THEN rate_per_100k END) AS male,
           MAX(CASE WHEN sex = 'Total'  THEN rate_per_100k END) AS total
    FROM cihi_children, latest
    WHERE service = 'hospitalisation' AND year = latest.y
    GROUP BY diagnosis_category
    ORDER BY total DESC
""")

,diagnosis_category,female,male,total
0,Neurocognitive disorders,799.1,573.9,738.6
1,Anxiety disorders,851.6,679.5,696.8
2,Substance-related disorders,823.0,610.1,690.5
3,Selected disorders diagnosed in childhood,846.0,611.4,667.5
4,Mood disorders,792.7,633.6,634.9


**Q26 · Fastest-growing diagnosis category** — ED visits, first vs latest fiscal year

In [30]:
q("""
    WITH agg AS (
        SELECT diagnosis_category, year, AVG(rate_per_100k) AS rate
        FROM cihi_children
        WHERE service = 'ED visit' AND sex = 'Total'
        GROUP BY diagnosis_category, year
    ),
    ranked AS (
        SELECT diagnosis_category, rate,
               ROW_NUMBER() OVER (PARTITION BY diagnosis_category ORDER BY year ASC)  AS r_first,
               ROW_NUMBER() OVER (PARTITION BY diagnosis_category ORDER BY year DESC) AS r_last
        FROM agg
    )
    SELECT f.diagnosis_category,
           ROUND(f.rate, 1) AS first_rate,
           ROUND(l.rate, 1) AS last_rate,
           ROUND(100.0 * (l.rate - f.rate) / NULLIF(f.rate, 0), 1) AS pct_change
    FROM ranked f JOIN ranked l ON f.diagnosis_category = l.diagnosis_category
    WHERE f.r_first = 1 AND l.r_last = 1
    ORDER BY pct_change DESC
""")

,diagnosis_category,first_rate,last_rate,pct_change
0,Anxiety disorders,1255.4,1486.7,18.4
1,Selected disorders diagnosed in childhood,1218.1,1414.2,16.1
2,Mood disorders,1243.8,1373.4,10.4
3,Substance-related disorders,1278.1,1389.7,8.7
4,Neurocognitive disorders,1325.8,1420.8,7.2


**Q27 · Age group with the highest rate** (ED visits, latest year, Total)

In [31]:
q("""
    WITH latest AS (SELECT MAX(year) y FROM cihi_children)
    SELECT age_group, ROUND(AVG(rate_per_100k), 1) AS avg_rate_per_100k
    FROM cihi_children, latest
    WHERE service = 'ED visit' AND sex = 'Total' AND year = latest.y
    GROUP BY age_group
    ORDER BY avg_rate_per_100k DESC
""")

,age_group,avg_rate_per_100k
0,15-17,2549.8
1,10-14,1356.6
2,5-9,344.4


## 10 · KPI development

Assemble the numbers the dashboard needs into one tidy table and export it to
`data/processed/04_kpi_summary.csv`. Each KPI links back to a query above.

In [32]:
kpi_sql = {
"national_fair_poor_mh_latest_pct": """
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator='Perceived mental health, fair or poor')
    SELECT value FROM mh_long, latest
    WHERE indicator='Perceived mental health, fair or poor'
      AND geo='Canada' AND sex='Both' AND year=latest.y
""",
"national_fair_poor_mh_change_pp": """
    WITH r AS (SELECT value,
                 ROW_NUMBER() OVER (ORDER BY year ASC)  rf,
                 ROW_NUMBER() OVER (ORDER BY year DESC) rl
               FROM mh_long
               WHERE indicator='Perceived mental health, fair or poor'
                 AND geo='Canada' AND sex='Both')
    SELECT ROUND(MAX(CASE WHEN rl=1 THEN value END) - MAX(CASE WHEN rf=1 THEN value END), 1) FROM r
""",
"highest_burden_province": """
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator='Perceived mental health, fair or poor')
    SELECT geo || ' (' || value || '%)' FROM mh_long, latest
    WHERE indicator='Perceived mental health, fair or poor' AND sex='Both'
      AND geo_level='province' AND year=latest.y AND value IS NOT NULL
    ORDER BY value DESC LIMIT 1
""",
"lowest_burden_province": """
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator='Perceived mental health, fair or poor')
    SELECT geo || ' (' || value || '%)' FROM mh_long, latest
    WHERE indicator='Perceived mental health, fair or poor' AND sex='Both'
      AND geo_level='province' AND year=latest.y AND value IS NOT NULL
    ORDER BY value ASC LIMIT 1
""",
"provinces_worsening_count": """
    WITH r AS (SELECT geo, value,
                 ROW_NUMBER() OVER (PARTITION BY geo ORDER BY year ASC)  rf,
                 ROW_NUMBER() OVER (PARTITION BY geo ORDER BY year DESC) rl
               FROM mh_long
               WHERE indicator='Perceived mental health, fair or poor'
                 AND sex='Both' AND geo_level='province')
    SELECT SUM(CASE WHEN last_val > first_val THEN 1 ELSE 0 END)
    FROM (SELECT f.geo, f.value first_val, l.value last_val
          FROM r f JOIN r l ON f.geo=l.geo WHERE f.rf=1 AND l.rl=1)
""",
"national_suicidal_thoughts_latest_pct": """
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator='Suicidal thoughts (15 years and over)')
    SELECT value FROM mh_long, latest
    WHERE indicator='Suicidal thoughts (15 years and over)'
      AND geo='Canada' AND sex='Both' AND year=latest.y
""",
"female_minus_male_fair_poor_mh_pp": """
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator='Perceived mental health, fair or poor')
    SELECT ROUND(MAX(CASE WHEN sex='Female' THEN value END)
               - MAX(CASE WHEN sex='Male'   THEN value END), 1)
    FROM mh_long, latest
    WHERE indicator='Perceived mental health, fair or poor'
      AND geo='Canada' AND year=latest.y
""",
"provincial_spread_fair_poor_mh_pp": """
    WITH latest AS (SELECT MAX(year) y FROM mh_long
                    WHERE indicator='Perceived mental health, fair or poor')
    SELECT ROUND(MAX(value) - MIN(value), 1) FROM mh_long, latest
    WHERE indicator='Perceived mental health, fair or poor' AND sex='Both'
      AND geo_level='province' AND year=latest.y AND value IS NOT NULL
""",
"help_seeking_ratio_national": """
    WITH latest AS (SELECT indicator, MAX(year) y FROM mh_long GROUP BY indicator)
    SELECT ROUND(
        (SELECT value FROM mh_long m JOIN latest l ON l.indicator=m.indicator AND l.y=m.year
         WHERE m.indicator='Consultation with a health professional about emotional or mental health'
           AND m.geo='Canada' AND m.sex='Both')
      / NULLIF((SELECT value FROM mh_long m JOIN latest l ON l.indicator=m.indicator AND l.y=m.year
         WHERE m.indicator='Suicidal thoughts (15 years and over)'
           AND m.geo='Canada' AND m.sex='Both'), 0), 2)
""",
"youth_ed_visit_rate_latest_per100k": """
    WITH latest AS (SELECT MAX(year) y FROM cihi_children)
    SELECT ROUND(AVG(rate_per_100k), 1) FROM cihi_children, latest
    WHERE service='ED visit' AND sex='Total' AND year=latest.y
""",
}

kpi_rows = []
for name, sql in kpi_sql.items():
    try:
        val = q(sql).iloc[0, 0]
    except Exception as e:
        val = f"ERROR: {e}"
    kpi_rows.append({"kpi": name, "value": val})

kpi_summary = pd.DataFrame(kpi_rows)
kpi_summary["data_source"] = "SAMPLE (not real)" if USING_SAMPLE else "data/processed"
kpi_summary


,kpi,value,data_source
0,national_fair_poor_mh_latest_pct,11.9,SAMPLE (not real)
1,national_fair_poor_mh_change_pp,0.7,SAMPLE (not real)
2,highest_burden_province,Manitoba (12.6%),SAMPLE (not real)
3,lowest_burden_province,Nova Scotia (11.1%),SAMPLE (not real)
4,provinces_worsening_count,6,SAMPLE (not real)
5,national_suicidal_thoughts_latest_pct,4.6,SAMPLE (not real)
6,female_minus_male_fair_poor_mh_pp,0.7,SAMPLE (not real)
7,provincial_spread_fair_poor_mh_pp,1.5,SAMPLE (not real)
8,help_seeking_ratio_national,2.04,SAMPLE (not real)
9,youth_ed_visit_rate_latest_per100k,1417.0,SAMPLE (not real)


In [33]:
out = PROCESSED / "04_kpi_summary.csv"
kpi_summary.to_csv(out, index=False)
print(("SAMPLE " if USING_SAMPLE else "") + "KPI summary written to", out.relative_to(ROOT))


SAMPLE KPI summary written to data/processed/04_kpi_summary.csv


## 11 · Notes for the team

- **Everything above runs on SAMPLE data** until `data/processed/mh_long.csv` and
  `cihi_children.csv` exist. The sample only has 2 sources and 3 periods — real coverage
  differs per source (see `docs/data_inventory.md`).
- If your cleaned column names differ from the contract, fill in the `COLS` map in section 2
  instead of editing every query.
- Indicator label strings must match exactly. Run the reference cell at the end of section 2
  to see what's actually loaded, then adjust the `WHERE indicator = '...'` clauses.
- `cchs_mh_disorders`, `stress_coping` and `perceived_health_quarterly` are in the contract
  but not in the sample — their queries slot into sections 4–8 the same way once loaded.
- KPI definitions and formulas belong in `docs/kpi_definitions.md` — this notebook is the
  calculation, that doc is the explanation.
- Close the DB when done: `con.close()`.
